# Cross-Sectional Factor Research Framework
## US Equities — S&P 500 Universe

This notebook provides an end-to-end walkthrough of the factor research pipeline:
1. Data loading (prices, market cap, fundamentals)
2. Factor construction (Momentum, Value, Quality, Low Vol, Reversal, Size)
3. IC / ICIR analysis
4. Factor decay curves
5. Quintile portfolio analysis
6. Fama-MacBeth regression
7. Factor correlation matrix
8. Long-short portfolio performance with ERC weights
9. Deflated Sharpe Ratio

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print('Dependencies loaded OK')

## 1. Data Loading

In [ ]:
from data import load_prices, load_market_cap, load_fundamentals_simulated, SP500_TICKERS

START = '2015-01-01'
END   = '2024-06-30'
# Use first 80 tickers for faster notebook execution; remove slice for full run
TICKERS = SP500_TICKERS[:80]

prices = load_prices(tickers=TICKERS, start=START, end=END)
print(f'Prices: {prices.shape[0]} days × {prices.shape[1]} tickers')
prices.tail(3)

In [ ]:
market_cap = load_market_cap(prices=prices, tickers=list(prices.columns), start=START, end=END)
fundamentals = load_fundamentals_simulated(prices)
print('Market cap shape:', market_cap.shape)
print('Fundamentals keys:', list(fundamentals.keys()))

## 2. Factor Construction

In [ ]:
from factors import (
    compute_momentum, compute_value, compute_quality,
    compute_low_volatility, compute_reversal, compute_size,
)

f_momentum = compute_momentum(prices)
f_value    = compute_value(market_cap, fundamentals['book_value'])
f_quality  = compute_quality(fundamentals['net_income'], fundamentals['book_equity'])
f_lowvol   = compute_low_volatility(prices)
f_reversal = compute_reversal(prices)
f_size     = compute_size(market_cap)

factor_dict = {
    'Momentum' : f_momentum,
    'Value'    : f_value,
    'Quality'  : f_quality,
    'LowVol'   : f_lowvol,
    'Reversal' : f_reversal,
    'Size'     : f_size,
}

print('Factors computed:')
for name, fdf in factor_dict.items():
    coverage = fdf.notna().sum(axis=1).mean()
    print(f'  {name:<12} shape={fdf.shape}  avg_coverage={coverage:.0f} stocks/day')

## 3. IC / ICIR Analysis

In [ ]:
from analytics import ic_summary

def fwd_ret(prices, h):
    return prices.shift(-h) / prices - 1

fwd_rets = {'1d': fwd_ret(prices, 1), '5d': fwd_ret(prices, 5), '20d': fwd_ret(prices, 20)}

ic_rows = []
for fname, fsig in factor_dict.items():
    s = ic_summary(fsig, fwd_rets, window=252)
    s.insert(0, 'factor', fname)
    ic_rows.append(s)

ic_df = pd.concat(ic_rows, ignore_index=True)
ic_df

In [ ]:
# Pivot: factors as rows, horizons as columns, mean_IC as values
ic_pivot = ic_df.pivot(index='factor', columns='horizon', values='mean_IC')
icir_pivot = ic_df.pivot(index='factor', columns='horizon', values='ICIR')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
import seaborn as sns

sns.heatmap(ic_pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Mean IC by Factor & Horizon', fontweight='bold')

sns.heatmap(icir_pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('ICIR by Factor & Horizon', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/ic_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Factor Decay Curves

In [ ]:
from analytics import factor_decay_curve
from analytics.factor_decay import plot_factor_decay

decay_df = factor_decay_curve(factor_dict, prices)
fig = plot_factor_decay(decay_df, save_path='../outputs/factor_decay.png')
plt.show()
print('Decay table:')
decay_df.round(4)

## 5. Quintile Portfolio Analysis

In [ ]:
from analytics import quintile_portfolios
from analytics.quintile_analysis import plot_quintile_spreads

quintile_dict = {}
for fname, fsig in factor_dict.items():
    quintile_dict[fname] = quintile_portfolios(fsig, prices)

fig = plot_quintile_spreads(quintile_dict, save_path='../outputs/quintile_spreads.png')
plt.show()

In [ ]:
# Q1-Q5 spread summary
spread_summary = pd.DataFrame({
    name: {
        'mean_Q1 (%)': df['Q1'].mean() * 100,
        'mean_Q5 (%)': df['Q5'].mean() * 100,
        'mean_spread (%)': df['spread'].mean() * 100,
        'spread_t_stat': df['spread'].mean() / (df['spread'].std() / len(df)**0.5)
    }
    for name, df in quintile_dict.items()
}).T
spread_summary.round(3)

## 6. Fama-MacBeth Regression

In [ ]:
from analytics import fama_macbeth_regression

monthly_ret = prices.resample('ME').last().pct_change()
monthly_factors = {name: sig.resample('ME').last() for name, sig in factor_dict.items()}

coef_ts, fm_summary = fama_macbeth_regression(monthly_factors, monthly_ret)
print('Fama-MacBeth Results:')
fm_summary

In [ ]:
# Plot time series of coefficients
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()
for i, col in enumerate(coef_ts.columns):
    ax = axes[i]
    coef_ts[col].rolling(6).mean().plot(ax=ax, color='steelblue', linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    t_stat = fm_summary.loc[col, 't_stat'] if col in fm_summary.index else float('nan')
    ax.set_title(f'{col} (t={t_stat:.2f})', fontsize=11, fontweight='bold')
    ax.set_ylabel('Coefficient')
    ax.grid(True, alpha=0.3)
plt.suptitle('Fama-MacBeth Coefficients (6m rolling mean)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/fama_macbeth_coefs.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Factor Correlation Matrix

In [ ]:
from analytics import factor_correlation_matrix

avg_corr, crowded_pairs = factor_correlation_matrix(factor_dict)

fig, ax = plt.subplots(figsize=(8, 6))
import seaborn as sns
mask = np.eye(len(avg_corr), dtype=bool)
sns.heatmap(avg_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, mask=mask)
ax.set_title('Average Monthly Cross-Sectional Factor Correlations\n(Spearman)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/factor_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

if not crowded_pairs.empty:
    print('\nCrowded pairs (|corr| > 0.6):')
    print(crowded_pairs.to_string(index=False))
else:
    print('No crowded factor pairs detected.')

## 8. Long-Short Portfolio Performance

In [ ]:
from portfolio import build_long_short_portfolio, portfolio_performance

port_df = build_long_short_portfolio(factor_dict, prices)
perf = portfolio_performance(port_df, ret_col='net_ret')

print('Performance Metrics:')
for k, v in perf.items():
    print(f'  {k:<30} {v:+.4f}' if isinstance(v, float) else f'  {k:<30} {v}')

In [ ]:
# Cumulative return chart
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

cum_net  = (1 + port_df['net_ret'].dropna()).cumprod()
cum_gross = (1 + port_df['gross_ret'].dropna()).cumprod()

axes[0].plot(cum_gross.index, cum_gross.values, label='Gross', color='steelblue', lw=1.5)
axes[0].plot(cum_net.index,  cum_net.values,   label='Net (after 5bps TC)', color='darkorange', lw=1.5)
axes[0].set_title('ERC Long-Short Portfolio — Cumulative Return', fontweight='bold')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Drawdown
roll_max = cum_net.cummax()
dd = (cum_net - roll_max) / roll_max
axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.4)
axes[1].set_title('Drawdown', fontweight='bold')
axes[1].set_ylabel('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/portfolio_cumulative_return.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Deflated Sharpe Ratio

In [ ]:
from analytics.deflated_sharpe import deflated_sharpe_ratio

rets = port_df['net_ret'].dropna().values
dsr_result = deflated_sharpe_ratio(rets, n_trials=6, periods_per_year=252)

print('Deflated Sharpe Ratio Analysis:')
print(f"  Observed SR (per-day):   {dsr_result['observed_sr']:.4f}")
print(f"  Annualized SR:           {dsr_result['annualized_sr']:.4f}")
print(f"  Benchmark SR (6 trials): {dsr_result['benchmark_sr']:.4f}")
print(f"  Deflated Sharpe Ratio:   {dsr_result['dsr']:.4f}")
print(f"  p-value (DSR):           {dsr_result['p_value']:.4f}")
print()
if dsr_result['dsr'] > 0.95:
    print('  => Strategy survives the DSR test at 5% significance.')
else:
    print('  => Strategy does NOT survive the DSR test — further validation needed.')

## 10. Results Summary

In [ ]:
# Combined results table
ic_20d = ic_df[ic_df['horizon'] == '20d'].set_index('factor')

results_table = pd.DataFrame({
    'Mean IC (20d)' : ic_20d['mean_IC'].round(4),
    'ICIR (20d)'    : ic_20d['ICIR'].round(3),
    'IC>0 (%)' : ic_20d['IC_positive_pct'].round(1),
    'FM t-stat'     : fm_summary['t_stat'].round(2) if not fm_summary.empty else 'n/a',
    'FM Significant': fm_summary['significant'] if not fm_summary.empty else 'n/a',
})

print('\n=== Factor Research Results Summary ===')
print(results_table.to_string())
print()
print(f'Portfolio Sharpe (Net):  {perf.get("sharpe_ratio", float("nan")):.3f}')
print(f'Portfolio Max Drawdown:  {perf.get("max_drawdown", float("nan")):.2%}')
print(f'Deflated Sharpe Ratio:   {dsr_result["dsr"]:.4f}')
results_table